In [ ]:
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index
import numpy as np

embedder = Embedder('models/Xenova/all-MiniLM-L6-v2')

## Q1. Embedding a query

In [ ]:
query1 = 'How does approximate nearest neighbor search work?'
v = embedder.encode(query1)
print('v[0] =', v[0])  # -0.02058203437252893
print('length =', len(v))

## Load Data

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda path: '/lessons/' in path,
)
documents = [file.parse() for file in reader.read()]
print('Total docs:', len(documents))

## Q2. Cosine similarity

In [ ]:
target = next(d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')
v_target = embedder.encode(target['content'])
similarity = v.dot(v_target)
print('Cosine similarity =', similarity)  # 0.361

## Q3. Chunking and search by hand

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
contents = [c['content'] for c in chunks]
X = embedder.encode_batch(contents)
scores = X.dot(v)
best_idx = scores.argmax()
print('Best chunk filename =', chunks[best_idx]['filename'])  # 07-sqlitesearch-vector.md

## Q4. Vector search with minsearch

In [ ]:
vs = VectorSearch()
vs.fit(X, chunks)
query4 = 'What metric do we use to evaluate a search engine?'
v4 = embedder.encode(query4)
results4 = vs.search(v4, num_results=5)
print('Q4 first result =', results4[0]['filename'])  # 04-evaluation/lessons/05-search-metrics.md

## Q5. Text search vs vector search

In [ ]:
idx = Index(text_fields=['content'], keyword_fields=['filename'])
idx.fit(chunks)
query5 = 'How do I store vectors in PostgreSQL?'
v5 = embedder.encode(query5)
vector_results5 = vs.search(v5, num_results=5)
text_results5 = idx.search(query5, num_results=5)
vector_files = set(r['filename'] for r in vector_results5)
text_files = set(r['filename'] for r in text_results5)
print('In vector but not text =', vector_files - text_files)  # 08-pgvector.md

## Q6. Hybrid search (RRF)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc['filename'], doc['start'])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query6 = 'How do I give the model access to tools?'
v6 = embedder.encode(query6)
vector_results6 = vs.search(v6, num_results=5)
text_results6 = idx.search(query6, num_results=5)
results6 = rrf([vector_results6, text_results6])
print('Q6 first result =', results6[0]['filename'])  # 13-function-calling.md